# Кейс 1

## [Ссылка на Kaggle](https://www.kaggle.com/competitions/playground-series-s4e1/data)
### Состав команды:
- **Абиев Марик**
- **Барабошкина Кристина**
- **Варфоломеев Константин**
- **Владынцев Сергей**
- **Мартыненко Алексей**
- **Подгорнов Владислав**
- **Сычёв Иван**

# Оглавление

- [EDA](#eda)
    - [Библиотеки](#подгружаем-необходимые-библиотеки)
    - [Загрузка данных](#загружаем-данные)
    - [Состав данных и базовая структура](#проверить-состав-данных-и-базовую-структуру)
    - [Анализ пропусков в данных](#оценить-пропуски-в-данных)
    - [Анализ базовой структуры признаков](#оценить-базовую-статистику-признаков)
    - [Анализ выбросов](#оценить-выбросы)
    - [Анализ мультиколлинеарности](#оценить-мультиколлинеарность)
    - [Анализ взаимосвязи предикторов и таргета](#проанализировать-взаимосвязь-признаков-с-таргетом)
    - [Анализ баланса классов](#проверить-баланс-классов-для-классификации)
    - [Анализ распределения числовых признаков](#проверить-распределения-числовых-признаков)
    - [Анализ категориальных признаков](#проверить-важные-категориальные-признаки)
    - [Итоговые вывод и рекомендации](#итоговые-выводы-и-рекомендации)
- [Data prerpocessing и Feature Engineering на основе EDA](#data-prerpocessing-и-feature-engineering-на-основе-eda)
  - [Определимя с ключевой метрикой](#Определимя-с-ключевой-метрикой)
  - [Выделим колонки для преобразования](#Выделим-колонки-для-преобразования)
  - [Подготовим функции для предобработки данных](#Подготовим-функции-для-предобработки-данных)
  - [Формируем пайплайн для бейзлайна](#Формируем-пайплайн-для-бейзлайна)
  - [Обучение моделей](#Обучение-моделей)
  - [Простроение baseline модели](#Простроение-baseline-модели)
- [Отбор моделей](#Отбор-моделей)
  - [Сформируем пайплайн для алгоритмов на основе деревьев](#Сформируем-пайплайн-для-алгоритмов-на-основе-деревьев)
  - [Обучим DecisionTreeClassifier](#Обучим-DecisionTreeClassifier)
  - [Обучим RandomForestClassifier](#Обучим-RandomForestClassifier)
  - [Обучим CatBoostClassifier](#Обучим-CatBoostClassifier)
  - [Итоговая таблица с метриками](#Итоговая-таблица-с-метриками)
- [Обучим лучшую модель](#Обучим-лучшую-модель)
  - [Подберем гиперпараметры для лучшей модели с помощью Optuna](#Подберем-гиперпараметры-для-лучшей-модели-с-помощью-Optuna)
  - [Построим classification_report на тесте для лучшей модели](#Построим-classification-report-на-тесте-для-лучшей-модели)
  - [Оценим важностить фичей с помощью SHAP и feature_importances](#Оценим-важностить-фичей-с-помощью-SHAP-и-feature-importances)
  - [Попробуем убрать малоинформативные фичи](#Попробуем-убрать-малоинформативные-фичи)  
- [Выводы](#Выводы)

# EDA

## Подгружаем необходимые библиотеки

In [ ]:
# Зависимости ноутбука ставятся через poetry:
#   poetry install --with eda

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import phik
from phik import report

from typing import Tuple, Dict

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, PowerTransformer, StandardScaler, FunctionTransformer, PolynomialFeatures
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold, GridSearchCV, LearningCurveDisplay
from sklearn.metrics import classification_report, confusion_matrix, make_scorer, recall_score, precision_score, f1_score, accuracy_score, roc_auc_score

from catboost import CatBoostClassifier

from sklearn.inspection import permutation_importance
from optuna.integration import OptunaSearchCV
from optuna.distributions import IntDistribution, FloatDistribution, CategoricalDistribution

import shap

## Загружаем данные

In [ ]:
data = pd.read_csv('../data/raw/train.csv')
data.head()

## Проверить состав данных и базовую структуру

In [ ]:
def data_shape(data):
    print('Количество строк в датасете:', data.shape[0])
    print('Количество столбцов в датасете:', data.shape[1])

In [ ]:
data_shape(data)

In [ ]:
data.dtypes

In [ ]:
data.describe()

**Датасет состоит из 165034 и 14 столбцов, из которых:**
- 1 таргет (Exited)
- 3 неинформативные фичи (id, CustomerId, Surname)
- 2 бинарные фичи (HasCrCard, IsActiveMember)
- 6 числовых фич (CreditScore, Age, Tenure, Balance, EstimatedSalary, NumOfProducts)
- 2 категориальные фичи (Geography, Gender)

**Удалим неинформативные фичи**

In [ ]:
features_to_keep = ['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure',
                   'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember',
                   'EstimatedSalary', 'Exited']

df = data[features_to_keep].copy()

In [ ]:
df.head()

## Оценить пропуски в данных

In [ ]:
df.isnull().sum()

**Пропуски отсутствуют**

## Оценить базовую статистику признаков

In [ ]:
df.describe()

In [ ]:
df['Geography'].value_counts()

In [ ]:
df['Gender'].value_counts()

**Рассмотрим числовые признаки:**
- CreditScore - имеет диапазон от 350 до 850. Фича - результат работы модели кредитного скоринга, где вероятности переводят в скор.
- Age - имеет диапазон от 18 до 92. Данные кажутся адекватными, нет очень мальниких или очень больших значений. Среднее и медиана практически равны, что может говорить о нормалььности распределение, но его мы оценим позже.
- Balance - диапазон от 0 до 250898. При этом медиана 0, а среднее 55478, что говорит о скошенности распределения. Можно подумать о логарифмировании.
- Tenure - срок обслуживания. Диапазон от 0 до 10. Среднее 5 и Медиана 5.
- EstimatedSalary - от 11 до 199992. Большая дисперсия в данных


**Рассмотрим бинарные признаки:**
- HasCrCard	- бинарный признак. Среднее 0.75, то есть у большей части есть кредитные карты
- IsActiveMember - бинаррный признак. Среднее 0.49, то есть кол-во активных и неактивных участников примерно одиакого
- Exited - таргет. Среднее 0.21. Есть дисбаланс, но не критичный

**Рассмотрим порядковые признаки:**
- NumOfProducts - диапазон от 0 до 4. Среднее 1.55, то есть  в средней у пользователей меньше 2-х продуктов

**Рассмотрим категориальные признаки:**
- Geography - объектов, принадлежащих к категории 'France' в три раза больше чем тех, что принадлежат 'Spain' и 'Germany', которых, в свою очередь, примерно равное количество. В целом кол-во категорий небольшое, можно рассмотреть One-Hot Encoding
- Gender - мужчин больше чем женщин. 2 категории - закодируем как 0 и 1.

## Оценить выбросы

In [ ]:
numeric = ['CreditScore', 'Age', 'Tenure', 'Balance', 'EstimatedSalary', 'NumOfProducts']

fig, axes = plt.subplots(3, 2, figsize=(16, 12))
fig.suptitle('Боксплоты для числовых признаков', fontsize=15, fontweight='bold')

axes = axes.ravel()

for i, feature in enumerate(numeric):
    axes[i].boxplot(df[feature])
    axes[i].set_title(f'Боксплот для {feature}', fontsize=11, fontweight='bold')
    axes[i].grid(alpha=0.3)


**Выводы:**
Выбросы есть только в фичах 'CreditScore', 'Age'. В случае с возрастом, выбросы, вероятнее всего, являются не ошибкой ввода, а несут под собой реальную информацию. Примем решение о том, как поступать с фичами после анализа их распределения в п.8

## Оценить мультиколлинеарность

### Построим Фик-матрицу, т.к. она позволяет анализировать корреляции в т.ч. между категориальными признаками и видеть нелинейные взаимосвязи

In [ ]:
phik_matrix = df.phik_matrix()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(phik_matrix, annot=True, cmap='RdYlBu_r', fmt='.2f',
            center=0, square=True)
plt.title('Матрица Phik корреляций (все типы признаков)', fontsize=16, pad=20)
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

**Выводы:**
- Ярко выраженных корреляций межу фичами нет. Есть умеренная корреляция между:
    - Geography и Balance
    - NumOfProducts и Balance


## Проанализировать взаимосвязь признаков с таргетом

**Анализируя корреляцию таргета и фичей можно увидеть, что есть 3 фичи, которые обладают умеренной корреляцией с таргеом:**
- Age
- NumOfProducts
- IsActiveMember

## Проверить баланс классов (для классификации)

In [ ]:
Exited_val = df.Exited.value_counts(normalize=True)

In [ ]:
plt.figure(figsize=(6, 4))
sns.barplot(x=Exited_val.index, y=Exited_val.values, palette=['lightblue', 'coral'])
plt.title('Распределение оттока клиентов', fontsize=14, fontweight='bold')
plt.xlabel('Отток (Exited)')
plt.ylabel('Доля клиентов')
plt.xticks([0, 1], ['Остались (0)', 'Ушли (1)'])

plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

**Выводы:**
- Отношение классов примерно 1 к 5 (дисбаланс есть, но умеренный)
- Можно ограничиться встроенными в модели параметрами для работы с дисбалансом (class_weight)

## Проверить распределения числовых признаков

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 10))
fig.suptitle('Гистограммы распределения для числовых признаков', fontsize=16, fontweight='bold')

axes = axes.ravel()

for i, feature in enumerate(numeric):
    axes[i].hist(df[feature], bins = 10)
    axes[i].set_title(f'Гистограмма распределения для {feature}', fontsize=12, fontweight='bold')
    axes[i].grid(alpha=0.3)

**Выводы:**
- Признак **CreditScore** имеет распределение близкое к нормальному. Если будем применять линейные модели, то необходимо применить какой-либо скейлер. Если деревья - можно оставить как есть
- Признак **Age** имеет выраженный правый хвост. К тому же есть выбросы. Если будем использовать линейные модели, можно применить преобразование Бокса-Кокса ("сжать" данные), если деревья - оставить как есть, так как выбросы, вероятнее всего, являются не ошибкой ввода, а несут под собой реальную информацию.
- Признак **Tenure** - почти равномерное распределение. Дискретная величина. Можно сделать категориальную переменную (новый, средняя продолжительность, старый). Или оставить как есть.
- Признак **Balance** - "Zero-Inflated" метрика. Необходимо создать бинарную фичу "HasBalance", которая покажет, нулевой баланс или нет.
- Признак **EstimatedSalary** - оставить как есть, выбросов нет, распределение не сильно скошено влево.

## Проверить важные категориальные признаки

**На основе п.3 предлагаем сделать следующие преобразования с категориальными признаками:**
- **Бинарные признаки:**
    - HasCrCard	- Оставить как есть
    - IsActiveMember - Оставить как есть
- **Порядковые признаки:**
    - NumOfProducts - Оставить как есть
- **Рассмотрим категориальные признаки:**
    - Geography - Рассмотреть One-Hot Encoding
    - Gender - Закодируем как 0 и 1



## Итоговые выводы и рекомендации

| Категория | Признак | Действие | Причина |
|------------|----------|----------|----------|
| Неинформативные признаки | id | Удалить | Технический Id строки |
| Неинформативные признаки | CustomerId | Удалить | Id контрагента |
| Неинформативные признаки | Surname | Удалить | Фамилия контрагента |
| Пропуски | нет признаков с пропусками | - | - |
| Мультиколлинеарность | нет признаков с ярко выраженной мультиколлинеарностью | - | - |
| Выбросы | age | Для линейных моделей преобразование Бокса-Кокса, если деревья - оставить как есть | Сильная асимметрия, выбросы справа |
| Выбросы | CreditScore | Для линейных моделей - скейлер. Если деревья - можно оставить как есть | Асимметрия, выбросы слева |
| Ненормальное распределение | Balance | Создать бинарную фичу "HasBalance", которая покажет, нулевой баланс или нет | Сильная асимметрия |
| Ненормальное распределение | EstimatedSalary | Оставить как есть если деревья. Если линейные модели - скейлер | Несильная асимметрия, выбросов нет|
| Ненормальное распределение | Tenure | Можно сделать категориальную переменную (новый, средняя продолжительность, старый). Или оставить как есть. Для линейных моделей можно использовать скейлер | Равномерное распределение |
| Бинарные | HasCrCard | Оставить как есть | - |
| Бинарные | IsActiveMember | Оставить как есть | - |
| Числовые |  NumOfProducts | Оставить как есть | - |
| Категориальные | Geography | Рассмотреть One-Hot Encoding | 3 категории |
| Бинарные |  Gender | Закодируем как 0 и 1 | В датасете указаны как 'male' и 'female'|

# Data prerpocessing и Feature Engineering на основе EDA

## Определимя с ключевой метрикой

| Критерий | Оптимизировать Recall | Оптимизировать Precision | Оптимизировать F1 |
| :--- | :--- | :--- | :--- |
| **Бизнес-контекст** | Дешевые вмешательства, высокая цена потери клиента | Дорогие вмешательства, ограниченные ресурсы | Универсальный сценарий, компромисс |
| **Цель** | Найти ВСЕХ отточников | Чтобы в списке для акции были ТОЛЬКО отточники | Найти баланс |
| **Риск** | Пропустить отток (False Negative) | Потратить ресурсы напрасно (False Positive) | Учитывает оба риска |

**Остановимся на recall, так как хотим видеть большинство отточных**

## Выделим колонки для преобразования

In [ ]:
columns_to_drop = ['id', 'CustomerId', 'Surname']
cat_cols = ['Geography', 'Gender']
scale_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 'EstimatedSalary', 'NumOfProducts']

## Подготовим функции для предобработки данных

In [ ]:
def drop_non_informative(df: pd.DataFrame, columns_to_drop: list) -> pd.DataFrame:

    '''
    Функция для удаления неинформативных признаков.

    Args:
        df (pd.DataFrame): Датафрейм, из которого нужно удалить колонки
        columns_to_drop (list): Список колонок для удаления

    Returns:
        pd.DataFrame: Датафрейм с удаленными колонками
    '''

    return df.drop(columns=columns_to_drop)



def add_has_balance(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Функция для создания признака HasBalance

    Args:
        df (pd.DataFrame): Датафрейм, к которому нужно добавить колонку

    Returns:
        pd.DataFrame: Датафрейм с добавленной колонкой
    '''
    df_copy = df.copy()
    df_copy['HasBalance'] = (df_copy['Balance'] > 0).astype(int)

    return df_copy

## Формируем пайплайн для бейзлайна

In [ ]:
drop_transformer = FunctionTransformer(drop_non_informative,
                                       kw_args={'columns_to_drop': columns_to_drop},
                                       validate=False
                                       )

add_has_balance_transformer = FunctionTransformer(add_has_balance,
                                                  validate=False
                                                  )

In [ ]:
numeric_no_power_pipeline = make_pipeline(
    SimpleImputer(strategy='median')
)

categorical_pipeline = make_pipeline(
    OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num_other', numeric_no_power_pipeline, scale_cols),
        ('cat', categorical_pipeline, cat_cols),
    ],
    remainder='passthrough',
    verbose_feature_names_out=True
)

baseline_pipeline = Pipeline([
    ('drop_cols', drop_transformer),
    ('add_has_balance', add_has_balance_transformer),
    ('preprocessing', preprocessor),
    ('poly_features', PolynomialFeatures(degree=2, include_bias=False)),
    ('final_scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=42))
])

# Обучение моделей

## Простроение baseline модели

In [ ]:
X = data.drop(columns=['Exited'])
y = data['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    stratify=y, random_state=42)

In [ ]:
scoring = {
    'recall': make_scorer(recall_score),
    'precision': make_scorer(precision_score),
    'f1': make_scorer(f1_score)
}

In [ ]:
kf = StratifiedKFold(n_splits=3,
                     shuffle=True,
                     random_state=52)

param_grid_base = {
    'poly_features__degree': [1, 2],
    'classifier__C': [0.001, 0.01, 0.1, 1],
    'classifier__penalty': ['elasticnet'],
    'classifier__solver': ['saga'],
    'classifier__l1_ratio': [0, 0.25, 0.5, 0.75, 1],
    'classifier__class_weight': ['balanced', None]
}

search_base = GridSearchCV(baseline_pipeline,
                      param_grid_base,
                      cv=kf,
                      scoring=scoring,
                      refit='recall',
                      return_train_score=True,
                      n_jobs=-1
                      )

search_base.fit(X_train, y_train)

In [ ]:
search_info_base = pd.DataFrame(search_base.cv_results_)

display_cols_base = ['mean_test_recall', 'mean_test_precision', 'mean_test_f1',
                     'mean_train_recall', 'mean_train_precision', 'mean_train_f1', 'rank_test_recall']

metrics_info_base = pd.DataFrame(search_info_base.loc[search_base.best_index_, display_cols_base]).reset_index()
metrics_info_base.columns = ['Метрика', 'Значение']
metrics_info_base.sort_values('Значение')

In [ ]:
search_base.best_params_

### Найдем время обучение лучшего пайплайна

In [ ]:
start_time = time.time()
search_base.best_estimator_.fit(X_train, y_train)
training_time = time.time() - start_time

In [ ]:
log_speed = pd.DataFrame({'Метрика': ['Скорость обучения (сек)'],
                        'Значение': [training_time]})

metrics_info_base = pd.concat([metrics_info_base, log_speed], ignore_index=True)

In [ ]:
metrics_info_base

## Отбор моделей

### Сформируем пайплайн для алгоритмов на основе деревьев

In [ ]:
tree_preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_pipeline, cat_cols)
    ],
    remainder='passthrough',
    verbose_feature_names_out=True
)

tree_pipeline = Pipeline([
    ('drop_cols', drop_transformer),
    ('add_has_balance', add_has_balance_transformer),
    ('preprocessing', tree_preprocessor),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

### Обучим DecisionTreeClassifier

In [ ]:
param_grid = {
    'classifier__max_depth': [3, 5, 7, 10, 20],
    'classifier__min_samples_split': [2, 5, 10, 20],
    'classifier__min_samples_leaf': [1, 2, 5, 10],
    'classifier__class_weight': ['balanced', None]
}

DecisionTreeSearch = GridSearchCV(tree_pipeline,
                      param_grid,
                      cv=kf,
                      scoring=scoring,
                      refit='recall',
                      return_train_score=True,
                      n_jobs=-1
                      )

DecisionTreeSearch.fit(X_train, y_train)

In [ ]:
DecisionTreeResults = pd.DataFrame(DecisionTreeSearch.cv_results_)
display_cols_tree = ['mean_test_recall', 'mean_test_precision', 'mean_test_f1',
                   'mean_train_recall', 'mean_train_precision', 'mean_train_f1', 'rank_test_recall']

DecisionTreeMetrics_info = pd.DataFrame(DecisionTreeResults.loc[DecisionTreeSearch.best_index_, display_cols_tree]).reset_index()
DecisionTreeMetrics_info.columns = ['Метрика', 'Значение']
DecisionTreeMetrics_info.sort_values('Значение')

### Оценим время обучение лучшего пайплайна

In [ ]:
startDecision_time = time.time()
DecisionTreeSearch.best_estimator_.fit(X_train, y_train)
trainingDecision_time = time.time() - startDecision_time

In [ ]:
DecisionTree_speed = pd.DataFrame({'Метрика': ['Скорость обучения (сек)'],
                        'Значение': [trainingDecision_time]})

DecisionTreeMetrics_info = pd.concat([DecisionTreeMetrics_info, DecisionTree_speed], ignore_index=True)

In [ ]:
DecisionTreeMetrics_info

### Обучим RandomForestClassifier

In [ ]:
tree_pipeline.set_params(classifier=RandomForestClassifier(random_state=42, verbose=0))

param_grid = {
    'classifier__n_estimators': [100, 200, 500],
    'classifier__max_depth':  [3, 5, 7],
    'classifier__min_samples_split': [2, 4, 10],
    'classifier__min_samples_leaf': [1, 2, 5],
    'classifier__class_weight': ['balanced']
}

RandomForestSearch = GridSearchCV(tree_pipeline,
                      param_grid,
                      cv=kf,
                      scoring=scoring,
                      refit='recall',
                      return_train_score=True,
                      n_jobs=-1
                      )

RandomForestSearch.fit(X_train, y_train)

In [ ]:
RandomForestResults = pd.DataFrame(RandomForestSearch.cv_results_)
display_cols_tree  = ['mean_test_recall', 'mean_test_precision', 'mean_test_f1',
                      'mean_train_recall', 'mean_train_precision', 'mean_train_f1', 'rank_test_recall']

RandomForestResults_info = pd.DataFrame(RandomForestResults.loc[RandomForestSearch.best_index_, display_cols_tree]).reset_index()
RandomForestResults_info.columns = ['Метрика', 'Значение']
RandomForestResults_info.sort_values('Значение')

### Оценим время обучение лучшего пайплайна

In [ ]:
startForest_time = time.time()
RandomForestSearch.best_estimator_.fit(X_train, y_train)
trainingForest_time = time.time() - startForest_time

In [ ]:
RandomForest_speed = pd.DataFrame({'Метрика': ['Скорость обучения (сек)'],
                                  'Значение': [trainingForest_time]})

RandomForestResults_info = pd.concat([RandomForestResults_info, RandomForest_speed], ignore_index=True)

In [ ]:
RandomForestResults_info

### Обучим CatBoostClassifier

In [ ]:
tree_pipeline.set_params(classifier=CatBoostClassifier(random_state=42, verbose=0))

param_grid = {
        'classifier__iterations': [100, 200, 500],
        'classifier__depth': [3, 5, 7],
        'classifier__l2_leaf_reg': [3, 5, 7],
        'classifier__auto_class_weights': ['Balanced', 'SqrtBalanced']

}

CatBoostSearch = GridSearchCV(tree_pipeline,
                        param_grid,
                        cv=kf,
                        scoring=scoring,
                        refit='recall',
                        return_train_score=True,
                        n_jobs=-1
                      )

CatBoostSearch.fit(X_train, y_train)

In [ ]:
CatBoostResults = pd.DataFrame(CatBoostSearch.cv_results_)
display_cols_tree = ['mean_test_recall', 'mean_test_precision', 'mean_test_f1',
                     'mean_train_recall', 'mean_train_precision', 'mean_train_f1', 'rank_test_recall']

CatBoostResults_info = pd.DataFrame(CatBoostResults.loc[CatBoostSearch.best_index_, display_cols_tree]).reset_index()
CatBoostResults_info.columns = ['Метрика', 'Значение']
CatBoostResults_info.sort_values('Значение')

In [ ]:
startCatBoost_time = time.time()
CatBoostSearch.best_estimator_.fit(X_train, y_train)
trainingCatBoost_time = time.time() - startCatBoost_time

In [ ]:
CatBoost_speed = pd.DataFrame({'Метрика': ['Скорость обучения (сек)'],
                                  'Значение': [trainingCatBoost_time]})

CatBoostResults_info = pd.concat([CatBoostResults_info, CatBoost_speed], ignore_index=True)

In [ ]:
CatBoostResults_info

### Итоговая таблица с метриками

In [ ]:
result = metrics_info_base.merge(
    DecisionTreeMetrics_info, on='Метрика', suffixes=('_log_reg', '_tree')
).merge(
    RandomForestResults_info, on='Метрика'
).merge(
    CatBoostResults_info, on='Метрика', suffixes=('_rf', '_catboost')
)

result.sort_values(by = 'Метрика')

**Ключевая метрика для выбора - recall**. При сопоставимости результатов также обращаем внимание на скорость обучения. В нашем случашя модель по этим параметрам - catboost


# Обучим лучшую модель

## Подберем гиперпараметры для лучшей модели с помощью Optuna

In [ ]:
tree_pipeline.set_params(classifier=CatBoostClassifier(random_state=42, verbose=0))

param_grid_cb = {
     'classifier__iterations': IntDistribution(100, 1000, step=100),
     'classifier__depth': IntDistribution(4, 12, step=2),
     'classifier__l2_leaf_reg': IntDistribution(2, 10, step=2),
     'classifier__auto_class_weights': CategoricalDistribution(['Balanced', 'SqrtBalanced'])
}
search_cb = OptunaSearchCV(tree_pipeline,
                          param_grid_cb,
                          cv=kf,
                          scoring='recall',
                          return_train_score=True,
                          n_jobs=-1,
                          random_state = 42,
                          verbose=0
                          )

search_cb.fit(X_train, y_train)

In [ ]:
CatBoostResults_best = pd.DataFrame(search_cb.cv_results_)
display_cols_tree = ['mean_test_score',
                     'mean_train_score']

CatBoostResults_best_info = pd.DataFrame(CatBoostResults_best.loc[search_cb.best_index_, display_cols_tree]).reset_index()
CatBoostResults_best_info.columns = ['Метрика', 'Значение']
CatBoostResults_best_info.sort_values('Метрика')

## Построим classification_report на тесте для лучшей модели

In [ ]:
search_cb.best_params_

In [ ]:
y_pred = search_cb.predict(X_test)

In [ ]:
print(classification_report(y_test, y_pred))

**Убедимся что нет переобучения**

In [ ]:
LearningCurveDisplay.from_estimator(
    search_cb.best_estimator_,
    X_train,
    y_train,
    cv=kf,
    scoring='recall',
    n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10),
    score_type='both',
    line_kw={'marker': 'o'},
    std_display_style='fill_between'
)
plt.title('Learning Curves - Best Model')
plt.show()

## Оценим важностить фичей с помощью SHAP, feature_importances

**Feature importances**

In [ ]:
best_pipe = search_cb.best_estimator_

preprocessor = best_pipe.named_steps['preprocessing']
feature_names = preprocessor.get_feature_names_out()

clf = best_pipe.named_steps['classifier']
importances = clf.feature_importances_

fi = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=fi.head(12), x='importance', y='feature')
plt.title('Top 10 Feature Importances - CatBoost')
plt.tight_layout()
plt.show()

**Описание важности предикторов по feature_importances:**
- Наиболее важным предиктором алгоритм считает NumOfProducts.
- На втором месте - Age.
- Другие важные предикторы - бинарные признаки, такие как IsActiveMember, Geography_Germany, Gender_Male, HasBalance.
- Также важным является один из числовых признаков - Balance.

**SHAP**

In [ ]:
X_trans = best_pipe[:-1].transform(X_test)

explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_trans)

shap.summary_plot(shap_values, X_trans, feature_names=feature_names)

**Описание важности предикторов по SHAP:**
- Как и в случае с feature_importances, для SHAP наиболее важным признаком оказался NumOfProducts. При этом высокие значения признака могут говорить как о положительном, так и отрицательном влиянии на таргет. Маленькие значения больше указывают на то, что объект будет отнесен к классу 1.
- Как и в случае с feature_importances, для SHAP вторым по важности признаком оказался Age. Как правило, чем выше возраст, тем больше вероятность отнесения объекта к классу 1.
- Как и в случае с feature_importances, для SHAP третьим и четвертым по важности признаками оказались Gender_Male и IsActiveMember соответственно. Если их значение = 1, то это уменньшает вероятность отнесения объекта к классу 1.

## Попробуем убрать малоинформативные фичи

In [ ]:
X_train_new = X_train.drop(columns = ['HasCrCard', 'EstimatedSalary', 'Tenure'])
X_test_new = X_test.drop(columns = ['HasCrCard', 'EstimatedSalary', 'Tenure'])

In [ ]:
search_cb.fit(X_train_new, y_train)

In [ ]:
CatBoostResults_best = pd.DataFrame(search_cb.cv_results_)
display_cols_tree = ['mean_test_score',
                     'mean_train_score']

CatBoostResults_best_info = pd.DataFrame(CatBoostResults_best.loc[search_cb.best_index_, display_cols_tree]).reset_index()
CatBoostResults_best_info.columns = ['Метрика', 'Значение']
CatBoostResults_best_info.sort_values('Метрика')

In [ ]:
y_pred_new = search_cb.predict(X_test_new)

Модель без неинформативных предикторов

In [ ]:
print(classification_report(y_test, y_pred_new))

Модель со всеми предикторами

In [ ]:
print(classification_report(y_test, y_pred))

Модель без неинформативных предикторов стала чуть лучше, оставим ее

In [ ]:
best_pipe = search_cb.best_estimator_

preprocessor = best_pipe.named_steps['preprocessing']
feature_names = preprocessor.get_feature_names_out()

clf = best_pipe.named_steps['classifier']
importances = clf.feature_importances_

fi = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=fi.head(12), x='importance', y='feature')
plt.title('Top 10 Feature Importances - CatBoost')
plt.tight_layout()
plt.show()

## Выводы

**1. Был проведен исследовательский анализ данных, в результате которого:**
- Были выбраны методы обработки данных для разных типов моделей
- Предложены новые фичи на основе существующих
- Предложены методы кодирования категориальных переменных
- Обнаружен дисбаланс классов и предложены методы работы с ним
- Оформлены итоговые рекомендации по дальнейшей обработке данных и работе с моделью

**2. Выбрали ключевую метрику - реколл**  
**3. Был построен пайплайн, в качестве которого выступила логичтическая регрессия**  
**4. Было построено несколько моделей и подобраны гиперпараметры:**
- Дерево решений (метод на основе деревьев)
- Случайный лес (ансамблевый метод, беггинг)
- Катбуст (ансамблевый метод, бустинг)  

**5. На основе сравнительной таблицы, в которой отражены ключевые метрики качества моделей, а также скорость обучения, выбла выбрана лучшая модель. В нашем случае - катбуст.**

**6. С помощью оптюны подобрали лучшие параметриы модели.**  
**7. С помощью SHAP, feature_importances оценили важность фичей**  
**8. Убрали неинформативные фичи, переобучили модель с новым подбором гиперпараметров. Итоговые метрики модели:**

```
   precision    recall  f1-score   support

           0       0.94      0.82      0.87     26023
           1       0.54      0.80      0.64      6984

    accuracy                           0.81     33007
   macro avg       0.74      0.81      0.76     33007
weighted avg       0.85      0.81      0.82     33007
```

Наиболее важным признаком оказался NumOfProducts. При этом высокие значения признака могут говорить как о положительном, так и отрицательном влиянии на таргет. Маленькие значения больше указывают на то, что объект будет отнесен к классу 1.
Вторым по важности признаком оказался Age. Как правило, чем выше возраст, тем больше вероятность отнесения объекта к классу 1.





